In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ==============================================================================
# Task 1: Load and filter datasets to the 20 Africa-relevant diseases
# ==============================================================================

# Define target diseases (cleaned names, exact casing to match raw data)
africa_20 = [
    "Malaria", "Typhoid", "Diabetes", "Bronchial Asthma", "Hypertension", "Migraine",
    "Chicken pox", "Jaundice", "Common Cold", "Heart attack", "Urinary tract infection",
    "Gastroenteritis", "Hypoglycemia", "Allergy", "Tuberculosis", "Pneumonia",
    "Arthritis", "Paralysis (brain hemorrhage)", "Drug Reaction", "AIDS"
]

# Load original datasets using robust relative paths (will run on any machine)
df_dataset = pd.read_csv("data/raw/dataset.csv")
df_symptoms = pd.read_csv("data/raw/symptom_Description.csv")
df_severity = pd.read_csv("data/raw/Symptom-severity.csv")
df_precaution = pd.read_csv("data/raw/symptom_precaution.csv")

# Strip disease names of trailing/leading whitespaces to avoid matching errors
df_dataset["Disease"] = df_dataset["Disease"].str.strip()
df_symptoms["Disease"] = df_symptoms["Disease"].str.strip()
df_precaution["Disease"] = df_precaution["Disease"].str.strip()

# Filter datasets for the 20 target diseases
df_dataset_20 = df_dataset[df_dataset["Disease"].isin(africa_20)]
df_symptoms_20 = df_symptoms[df_symptoms["Disease"].isin(africa_20)]
df_precaution_20 = df_precaution[df_precaution["Disease"].isin(africa_20)]

# For severity: get the unique symptoms present in the filtered patient dataset
symptom_cols = df_dataset_20.columns[1:]
unique_symptoms = pd.Series(df_dataset_20[symptom_cols].values.ravel()).str.strip().dropna().unique()

# Clean symptoms in severity and filter
df_severity["Symptom"] = df_severity["Symptom"].str.strip()
df_severity_20 = df_severity[df_severity["Symptom"].isin(unique_symptoms)]

# Save filtered datasets to output files
df_dataset_20.to_csv("data/processed/dataset_Africa20.csv", index=False)
df_symptoms_20.to_csv("symptoms_Africa20.csv", index=False)
df_severity_20.to_csv("severity_Africa20.csv", index=False)
df_precaution_20.to_csv("precaution_Africa20.csv", index=False)

print(f"Task 1 Complete! Filtered files created successfully:")
print(f" - Patients dataset: {df_dataset_20.shape}")
print(f" - Descriptions: {df_symptoms_20.shape}")
print(f" - Precautions: {df_precaution_20.shape}")
print(f" - Severities: {df_severity_20.shape}")

## Task 2: Data Cleaning 



In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Define standard cleaning function for symptom strings
def clean_symptom(s):
    if not isinstance(s, str):
        return s
    # Lowercase, strip whitespaces, and replace spaces with underscores
    s = s.strip().lower().replace(" ", "_")
    # Replace multiple consecutive underscores with a single underscore
    s = re.sub(r'_+', '_', s)
    # Standardize specific spelling mismatches
    if s == 'foul_smell_ofurine':
        s = 'foul_smell_of_urine'
    return s

# 2. Load the filtered datasets
df_dataset = pd.read_csv("data/processed/dataset_Africa20.csv")
df_severity_raw = pd.read_csv("data/raw/Symptom-severity.csv")
df_symptoms = pd.read_csv("symptoms_Africa20.csv")
df_precaution = pd.read_csv("precaution_Africa20.csv")

# 3. Clean symptom columns in the patient dataset
symptom_cols = df_dataset.columns[1:]
df_dataset[symptom_cols] = df_dataset[symptom_cols].map(clean_symptom)

# 4. Clean disease names consistently in all datasets
df_dataset["Disease"] = df_dataset["Disease"].str.strip()
df_symptoms["Disease"] = df_symptoms["Disease"].str.strip()
df_precaution["Disease"] = df_precaution["Disease"].str.strip()

# 5. Clean severity dataset symptoms and refilter based on the unique symptoms in patient dataset
df_severity_raw["Symptom"] = df_severity_raw["Symptom"].map(clean_symptom)
unique_cleaned_symptoms = pd.Series(df_dataset[symptom_cols].values.ravel()).dropna().unique()
df_severity_clean = df_severity_raw[df_severity_raw["Symptom"].isin(unique_cleaned_symptoms)].drop_duplicates()

# 6. Report on duplicate records in the patient dataset
total_rows = len(df_dataset)
duplicate_rows = df_dataset.duplicated().sum()
print(f"Total patient records: {total_rows}")
print(f"Duplicate records found: {duplicate_rows} ({duplicate_rows / total_rows * 100:.2f}%)")

# Drop duplicates to avoid data leakage and model overfitting
df_dataset_clean = df_dataset.drop_duplicates()
print(f"Patient records after dropping duplicates: {len(df_dataset_clean)}")

# 7. Report on missing values
print("\nMissing values count in patient columns:")
print(df_dataset_clean.isnull().sum())

# 8. Save the cleaned and processed datasets back
df_dataset_clean.to_csv("data/processed/dataset_Africa20.csv", index=False)
df_severity_clean.to_csv("severity_Africa20.csv", index=False)

print("\nTask 2 Complete! Cleaned files saved successfully:")
print(f" - Cleaned Patients dataset: {df_dataset_clean.shape}")
print(f" - Cleaned Severities dataset (should contain all 80 symptoms): {df_severity_clean.shape}")

In [21]:
df_dataset_clean['Disease'].value_counts()

Disease
Chicken pox                     10
Migraine                        10
Diabetes                         9
Jaundice                         9
Pneumonia                        9
Common Cold                      9
Tuberculosis                     9
Hypoglycemia                     9
Typhoid                          9
Malaria                          8
Bronchial Asthma                 7
Drug Reaction                    6
Hypertension                     6
Arthritis                        6
AIDS                             5
Allergy                          5
Paralysis (brain hemorrhage)     5
Gastroenteritis                  5
Heart attack                     5
Urinary tract infection          5
Name: count, dtype: int64

In [23]:
len(df_dataset_clean)

146